In [ ]:
# ==================================================================================
# 🚀 AGENT IA V44 : L'ARCHITECTURE "SÉPARATION DES POUVOIRS" (VALIDÉE)
# ==================================================================================

# --- 1. INSTALLATION ---
import os
print("⏳ Installation des librairies... (1 minute)")
!pip install -q odfpy python-pptx python-docx docx2txt reportlab pdf2image openpyxl xlsxwriter pymupdf4llm "smolagents[litellm]" langchain langchain-community langchain-google-genai langchain-text-splitters duckduckgo-search faiss-cpu
# 🛠️ RÉPARATION DES DÉPENDANCES (HuggingFace)
print("⏳ Réparation des librairies en cours...")
!pip install -U sentence-transformers huggingface_hub langchain-community > /dev/null 2>&1
print("✅ Réparation terminée. MAINTENANT : REDÉMARRE LA SESSION (Menu 'Exécution' > 'Redémarrer la session').")
!apt-get install -y poppler-utils libreoffice-core libreoffice-writer libreoffice-calc libreoffice-impress > /dev/null 2>&1
print("✅ Installation terminée !")

In [ ]:
# ==================================================================================
# 🛠️ PATCH V47 : DÉMARRAGE HYBRIDE CORRIGÉ (IMPORT HUGGINGFACE OK)
# ==================================================================================

# --- 2. CONFIGURATION & IMPORTS ---
import time
import pandas as pd
import zipfile
import xml.etree.ElementTree as ET
import google.generativeai as genai
import subprocess
import io, warnings, csv, re
import docx2txt
import pymupdf4llm
import openpyxl
import xlsxwriter
import os

from smolagents import CodeAgent, LiteLLMModel, tool, DuckDuckGoSearchTool
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# ✅ L'IMPORT CRITIQUE QUI MANQUAIT :
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownTextSplitter
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from pptx import Presentation
from pptx.util import Inches, Pt
from docx import Document
from docx.shared import Inches as DocxInches
from docx.shared import Pt as DocxPt, RGBColor as DocxRGBColor
from PIL import Image
from reportlab.pdfgen import canvas
try: from pdf2image import convert_from_path
except: pass

warnings.filterwarnings("ignore")

In [ ]:
# --- 🔐 TA CLÉ API (SÉCURISÉE) ---
# Mets ta clé ici une bonne fois pour toutes
CLE_GOOGLE_VISION = "Votre_CLEE"

if CLE_GOOGLE_VISION == "Votre_CLEE":
    from getpass import getpass
    print("🔐 Clé non trouvée dans le code. Entre-la ci-dessous :")
    CLE_GOOGLE_VISION = getpass()

genai.configure(api_key=CLE_GOOGLE_VISION)
os.environ["GOOGLE_API_KEY"] = CLE_GOOGLE_VISION

In [ ]:
# --- 3. LE CERVEAU & LA MÉMOIRE (AUTO-RÉPARATION TOTALE) ---
print("🔌 Connexion au Cerveau (Gemini)...")
model_agent = LiteLLMModel(model_id="gemini/gemini-2.5-flash", api_key=CLE_GOOGLE_VISION, temperature=0.1)
print("✅ Cerveau Connecté.")

print("📥 Initialisation de la Mémoire...")

embeddings = None

# TENTATIVE 1 : GOOGLE (Peut échouer en 404)
try:
    print("   👉 Tentative Google Embedding (004)...")
    test_emb = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
    test_emb.embed_query("test")
    embeddings = test_emb
    print("✅ SUCCÈS : Mode Google activé.")
except Exception as e:
    print(f"   ❌ Échec Google (Erreur 404/API). Passage au Plan B.")

# TENTATIVE 2 : LOCAL (HUGGING FACE) - Infaillible
if embeddings is None:
    print("   👉 Activation du Plan B : Mémoire Locale (HuggingFace)...")
    try:
        # Modèle léger et très performant, gratuit, sans clé
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        print("✅ SUCCÈS : Mode Local (HuggingFace) activé. C'est plus robuste !")
    except Exception as e:
        raise Exception(f"🛑 CRITIQUE : Impossible de charger la mémoire locale. Erreur : {e}")

if embeddings is None:
    raise Exception("🛑 CRITIQUE : Échec total des mémoires.")

print("✅ Mémoire Active et Prête.")

In [ ]:
# ==============================================================================
# 4. MOTEURS DE LECTURE
# ==============================================================================

def appel_gemini_securise(prompt, image=None):
    """Appel Gemini Vision sécurisé avec RETRY et 2.5."""
    max_retries = 6; wait_time = 10
    try: model = genai.GenerativeModel("gemini-2.5-flash")
    except: model = genai.GenerativeModel("gemini-2.0-flash")

    for i in range(max_retries):
        try:
            if image: return model.generate_content([prompt, image]).text
            else: return model.generate_content(prompt).text
        except Exception as e:
            if "429" in str(e): time.sleep(wait_time); wait_time *= 2
            elif "404" in str(e): model = genai.GenerativeModel("gemini-1.5-flash")
            else: return f"Erreur Vision : {e}"
    return "Échec Quota."

def convertir_tout_document(chemin_fichier):
    """Dispatcher Universel"""
    if not os.path.exists(chemin_fichier): return ""
    ext = os.path.splitext(chemin_fichier)[1].lower()
    texte = ""
    print(f"📂 Lecture ({ext}) : {os.path.basename(chemin_fichier)}")
    try:
        if ext == ".pdf": texte = pymupdf4llm.to_markdown(chemin_fichier)
        elif ext in [".jpg", ".png", ".jpeg"]:
            img = Image.open(chemin_fichier)
            texte = appel_gemini_securise("Décris cette image en détail.", img)
        elif ext == ".docx": texte = docx2txt.process(chemin_fichier)
        elif ext == ".pptx":
            prs = Presentation(chemin_fichier)
            for slide in prs.slides:
                for shape in slide.shapes:
                    if hasattr(shape, "text"): texte += shape.text + "\n"
        elif ext in [".xlsx", ".xls"]:
            try: texte = pd.read_excel(chemin_fichier).fillna("").to_markdown(index=False)
            except: pass
        elif ext == ".txt":
            with open(chemin_fichier, 'r', encoding='utf-8', errors='ignore') as f: texte = f.read()
        elif ext == ".csv":
            try: texte = pd.read_csv(chemin_fichier).to_markdown(index=False)
            except: pass
    except Exception as e: return f"Erreur lecture globale: {e}"
    return texte

In [ ]:
# --- RAG ---
vectorstore_global = None
def initialiser_rag(fichiers):
    global vectorstore_global
    text_data = ""
    for f in fichiers: text_data += convertir_tout_document(f) + "\n\n"
    if not text_data.strip(): return
    chunks = MarkdownTextSplitter(chunk_size=1000).split_text(text_data)
    vectorstore_global = FAISS.from_texts(chunks, embeddings)
    print("✅ Mémoire chargée.")

In [ ]:
# ==============================================================================
# 5. OUTILS D'ACTION SPÉCIALISÉS (ARCHITECTURE V44)
# ==============================================================================

# --- A. OUTILS WORD ---

@tool
def createur_word(nom_fichier: str, contenu: str, style: str = "NORMAL", chemin_image: str = None) -> str:
    """
    Crée un NOUVEAU fichier Word (.docx).
    Args:
        nom_fichier: Nom du fichier à créer.
        contenu: Le texte à écrire.
        style: 'TITRE', 'GRAS', 'TAILLE_24', 'ROUGE', 'NORMAL'.
        chemin_image: Chemin d'une image à insérer (optionnel).
    """
    try:
        doc = Document()
        p = doc.add_paragraph()
        run = p.add_run(contenu)
        s = style.upper()
        if "GRAS" in s: run.bold = True
        if "ROUGE" in s: run.font.color.rgb = DocxRGBColor(255, 0, 0)
        if "TITRE" in s: p.style = 'Heading 1'
        if "TAILLE" in s:
            try: run.font.size = DocxPt(int(re.search(r'\d+', s).group()))
            except: pass
        if chemin_image: doc.add_picture(chemin_image, width=DocxInches(4))
        doc.save(nom_fichier)
        return "Word Créé."
    except Exception as e: return f"Erreur Création Word: {e}"

@tool
def modificateur_word(nom_fichier: str, texte_ancrage: str, texte_remplacement: str, action: str = "AJOUTER_FIN", chemin_image: str = None) -> str:
    """
    Modifie un fichier Word existant.
    Args:
        nom_fichier: Le fichier à modifier.
        texte_ancrage: Le texte à chercher (pour remplacement).
        texte_remplacement: Le nouveau texte (ou texte à ajouter).
        action: 'AJOUTER_FIN', 'REMPLACER'.
        chemin_image: Image à ajouter.
    """
    try:
        doc = Document(nom_fichier)
        if action == "REMPLACER":
            for p in doc.paragraphs:
                if texte_ancrage and texte_ancrage in p.text:
                    p.text = p.text.replace(texte_ancrage, texte_remplacement)
        else:
            doc.add_paragraph(texte_remplacement)
        if chemin_image: doc.add_picture(chemin_image, width=DocxInches(4))
        doc.save(nom_fichier)
        return "Word Modifié."
    except Exception as e: return f"Erreur Modif Word: {e}"

In [ ]:
# --- B. OUTILS EXCEL ---

@tool
def createur_excel(nom_fichier: str, nom_feuille: str = "Donnees", donnees_initiales: str = "") -> str:
    """
    Crée un NOUVEAU fichier Excel (.xlsx).
    Args:
        nom_fichier: Nom du fichier.
        nom_feuille: Nom de la première feuille.
        donnees_initiales: (Optionnel) Données CSV brutes à écrire.
    """
    try:
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = nom_feuille
        if donnees_initiales:
            rows = donnees_initiales.split('\n')
            for r_idx, row in enumerate(rows, 1):
                cols = row.split(',')
                for c_idx, val in enumerate(cols, 1):
                    ws.cell(row=r_idx, column=c_idx, value=val.strip())
        wb.save(nom_fichier)
        return f"Excel créé : {nom_fichier}"
    except Exception as e: return f"Erreur Création Excel: {e}"

@tool
def modificateur_excel(nom_fichier: str, cible: str, valeur: str, action: str = "ECRIRE", style: str = None) -> str:
    """
    Modifie un Excel existant.
    Args:
        nom_fichier: Le fichier Excel.
        cible: La cellule (ex: 'B2') ou plage (ex: 'A1:C1').
        valeur: La donnée à écrire.
        action: 'ECRIRE', 'FUSION'.
        style: 'ROUGE', 'JAUNE', 'GRAS', 'TITRE'.
    """
    try:
        wb = openpyxl.load_workbook(nom_fichier)
        ws = wb.active
        val_clean = valeur
        try: val_clean = float(valeur.replace(' ','').replace('€','').replace(',','.'))
        except: pass

        if action == "ECRIRE": ws[cible] = val_clean
        elif action == "FUSION":
            ws.merge_cells(cible)
            top_left = cible.split(':')[0]
            ws[top_left] = val_clean
            ws[top_left].alignment = Alignment(horizontal='center', vertical='center')

        c = ws[cible.split(':')[0]]
        if style == "GRAS": c.font = Font(bold=True)
        if style == "ROUGE": c.font = Font(color="FF0000")
        if style == "JAUNE": c.fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        if style == "TITRE":
             c.font = Font(bold=True, color="FFFFFF", size=12)
             c.fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
        wb.save(nom_fichier)
        return "Excel Modifié."
    except Exception as e: return f"Erreur Modif Excel: {e}"

In [ ]:
# --- C. OUTILS POWERPOINT (RECONSTRUCTION INTELLIGENTE) ---

@tool
def createur_ppt(nom_fichier: str, texte: str, chemin_image: str = None) -> str:
    """
    Crée un PowerPoint avec la logique "Flux Vertical" : Image -> Espace -> Texte.
    Gère automatiquement la création de nouvelles diapositives si le contenu déborde.
    Args:
        nom_fichier: Nom du fichier.
        texte: Texte long à insérer.
        chemin_image: Image à insérer (optionnel).
    """
    try:
        if os.path.exists(nom_fichier): os.remove(nom_fichier)
        prs = Presentation()
        slide = prs.slides.add_slide(prs.slide_layouts[1])
        if slide.shapes.title: slide.shapes.title.text = "Présentation IA"
        cursor_y = Inches(1.5)

        # 1. IMAGE
        if chemin_image:
            img_height = Inches(3.5)
            slide.shapes.add_picture(chemin_image, Inches(1), cursor_y, height=img_height)
            cursor_y += img_height + Inches(0.2)

        # 2. TEXTE
        if texte:
            paragraphes = texte.split('\n')
            tb = slide.shapes.add_textbox(Inches(1), cursor_y, Inches(8), Inches(1))
            tf = tb.text_frame
            tf.word_wrap = True

            for para in paragraphes:
                if not para.strip(): continue
                lines = max(1, len(para) / 90)
                height_needed = lines * 0.3

                if (cursor_y / Inches(1)) + height_needed > 7.5:
                    slide = prs.slides.add_slide(prs.slide_layouts[5])
                    cursor_y = Inches(1)
                    tb = slide.shapes.add_textbox(Inches(1), cursor_y, Inches(8), Inches(1))
                    tf = tb.text_frame
                    tf.word_wrap = True

                p = tf.add_paragraph()
                p.text = para
                p.font.size = Pt(12)
                cursor_y += Inches(height_needed)

        prs.save(nom_fichier)
        return "PPT Créé (Flux continu respecté)."
    except Exception as e: return f"Erreur Création PPT: {e}"

@tool
def modificateur_ppt(nom_fichier: str, numero_slide: str, nouveau_texte: str, chemin_nouvelle_image: str = None) -> str:
    """
    Modifie une diapo en la RECONSTRUISANT proprement pour éviter les chevauchements.
    Si le texte déborde, crée de nouvelles slides.
    Args:
        nom_fichier: Le fichier PPTX.
        numero_slide: Numéro de la diapo.
        nouveau_texte: Le texte final à mettre.
        chemin_nouvelle_image: (Optionnel) Nouvelle image à mettre.
    """
    try:
        prs = Presentation(nom_fichier)
        idx = int(numero_slide) - 1
        slide = prs.slides[idx]
        # Nettoyage
        for shape in list(slide.shapes):
            if not shape == slide.shapes.title:
                sp = shape._element
                sp.getparent().remove(sp)

        # Reconstruction
        cursor_y = Inches(1.5)
        if chemin_nouvelle_image:
            img_height = Inches(3.5)
            slide.shapes.add_picture(chemin_nouvelle_image, Inches(1), cursor_y, height=img_height)
            cursor_y += img_height + Inches(0.2)

        if nouveau_texte:
            paragraphes = nouveau_texte.split('\n')
            tb = slide.shapes.add_textbox(Inches(1), cursor_y, Inches(8), Inches(1))
            tf = tb.text_frame
            tf.word_wrap = True

            for para in paragraphes:
                if not para.strip(): continue
                lines = max(1, len(para) / 90)
                height_needed = lines * 0.3

                if (cursor_y / Inches(1)) + height_needed > 7.5:
                    new_slide = prs.slides.add_slide(prs.slide_layouts[5])
                    cursor_y = Inches(1)
                    tb = new_slide.shapes.add_textbox(Inches(1), cursor_y, Inches(8), Inches(1))
                    tf = tb.text_frame
                    tf.word_wrap = True

                p = tf.add_paragraph()
                p.text = para
                p.font.size = Pt(12)
                cursor_y += Inches(height_needed)

        prs.save(nom_fichier)
        return "PPT Modifié (Slide reconstruite + Overflow géré)."
    except Exception as e: return f"Erreur Modif PPT: {e}"

In [ ]:
# --- D. OUTILS DIVERS (TXT/CSV, PDF, VISION) ---

@tool
def editeur_texte_csv(nom_fichier: str, contenu: str, mode: str = "AJOUTER_FIN") -> str:
    """
    Crée ou modifie un fichier texte (.txt) ou CSV (.csv).

    Args:
        nom_fichier: Le nom du fichier.
        contenu: Le texte à écrire.
        mode: 'AJOUTER_FIN' (ajoute à la fin) ou 'ECRASER' (remplace tout).
    """
    try:
        m = 'w' if mode=="ECRASER" else 'a'
        with open(nom_fichier, m, encoding='utf-8') as f: f.write("\n"+contenu)
        return "Fichier Texte Modifié."
    except Exception as e: return f"Erreur TXT: {e}"

@tool
def convertisseur_pdf_vers_editable(chemin_source: str, type_sortie: str = "docx") -> str:
    """
    Convertit PDF -> Editable (docx ou xlsx).

    Args:
        chemin_source: Le fichier PDF.
        type_sortie: 'docx' ou 'xlsx'.
    """
    try:
        nom_sortie = os.path.splitext(chemin_source)[0] + "." + type_sortie
        abs_in = os.path.abspath(chemin_source)
        args = ['libreoffice', '--headless', '--convert-to', type_sortie, abs_in, '--outdir', '.']
        if type_sortie == 'xlsx': args.append('--infilter=CSV:44,34,76')
        subprocess.run(args, check=True, stdout=subprocess.DEVNULL)
        return f"Succès. Fichier converti en : {nom_sortie}."
    except Exception as e: return f"Erreur Conversion: {e}"

@tool
def convertisseur_editable_vers_pdf(chemin_source: str) -> str:
    """
    Convertit Editable -> PDF.

    Args:
        chemin_source: Fichier DOCX ou XLSX.
    """
    try:
        abs_in = os.path.abspath(chemin_source)
        subprocess.run(['libreoffice', '--headless', '--convert-to', 'pdf', abs_in, '--outdir', '.'], check=True, stdout=subprocess.DEVNULL)
        return "Succès. Reconverti en PDF."
    except Exception as e: return f"Erreur Conversion: {e}"

@tool
def outil_vision(chemin_image: str, question: str) -> str:
    """
    Analyse image avec Gemini.

    Args:
        chemin_image: Chemin de l'image.
        question: Question à poser.
    """
    try: return appel_gemini_securise(f"{question}", Image.open(chemin_image))
    except: return "Erreur Vision"

@tool
def outil_rag(question: str) -> str:
    """
    Cherche info textuelle dans la mémoire (RAG).

    Args:
        question: La question.
    """
    global vectorstore_global
    if vectorstore_global is None: return "Aucun document."
    try:
        docs = vectorstore_global.similarity_search(question, k=10)
        return "\n".join([d.page_content for d in docs])
    except: return "Erreur RAG"

@tool
def web_search(query: str) -> str:
    """
    Recherche Web (DuckDuckGo).

    Args:
        query: Mots-clés.
    """
    try: return DuckDuckGoSearchTool().run(query)
    except: return "Erreur Web"

In [ ]:
# ASSEMBLAGE V44 (COMPLET)
liste_outils = [outil_rag, outil_vision, createur_word, modificateur_word, createur_excel, modificateur_excel, createur_ppt, modificateur_ppt, editeur_texte_csv, convertisseur_pdf_vers_editable, convertisseur_editable_vers_pdf, web_search]
imports = ["os", "pandas", "zipfile", "openpyxl", "pptx", "docx", "subprocess", "reportlab", "PIL", "csv", "pdf2image", "re"]

agent = CodeAgent(
    model=model_agent,
    tools=liste_outils,
    additional_authorized_imports=imports,
    max_steps=20
)

consigne = """
RÈGLES D'OR V44 :
1. ANALYSE LA DEMANDE : Word? Excel? PPT? TXT?
2. MODIF PPT : Utilise 'modificateur_ppt'. Il vide la slide et la recrée proprement avec le contenu final (Image + Texte + Overflow).
3. MODIF PDF : Convertis (docx/xlsx) -> Modifie -> Reconvertis.
4. HONNÊTETÉ : Si tu ne trouves pas une info, dis "NON TROUVÉE".
"""
agent.prompt_templates["system_prompt"] = consigne + agent.prompt_templates["system_prompt"]

print("✅ AGENT V44 OPÉRATIONNEL (ARCHITECTURE VALIDÉE).")

#MISSIONS

In [ ]:
# Mission Audit
# 1. On charge le fichier généré
fichiers_mission = ['rapport_financier_fictif.pdf']
initialiser_rag(fichiers_mission)

# 2. La Mission
mission_audit = """
Agis comme un Auditeur Financier.
Analyse le document PDF 'rapport_financier_fictif.pdf'.

TACHE 1 : Extraction
Trouve :
- Le Chiffre d'Affaires.
- Le Résultat Net.
- Le nom du PDG.
- Le Ratio de Solvabilité (caché dans le texte).

TACHE 2 : Création Excel
Crée un fichier 'Audit_TechNova.xlsx'.
1. Crée les colonnes 'Indicateur' et 'Valeur'.
2. Remplis avec les données trouvées.
3. Mets la ligne d'en-tête (Ligne 1) en GRAS, ROUGE et CENTRÉ.
"""

print(f"🎯 MISSION AUDIT : {mission_audit}")
try:
    res = agent.run(mission_audit)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission LibreOffice ODT/ODP
initialiser_rag([])

mission_libreoffice = """
Agis comme un Secrétaire sous Linux.

1. Crée un nouveau document texte au format LibreOffice (.odt) nommé 'Projet_Libre.odt'.
2. Écris dedans : "Ce fichier a été généré et modifié en format OpenDocument."
3. Applique le style GRAS et TAILLE:24 sur cette phrase.

4. Crée une présentation LibreOffice (.odp) nommée 'Diapo_Libre.odp'.
5. Ajoute une slide avec le titre "Vive l'Open Source".

IMPORTANT : Je veux récupérer des fichiers .odt et .odp à la fin.
"""

print(f"🎯 MISSION LIBREOFFICE : {mission_libreoffice}")
try:
    res = agent.run(mission_libreoffice)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission Vision
# ⚠️ REMPLACE PAR LE NOM DE TA PHOTO ⚠️
nom_fichier_image = 'Gemini_Generated_Image_dg9rkvdg9rkvdg9r.png'

initialiser_rag([nom_fichier_image])

mission_vision = f"""
Agis comme un Archiviste.
Regarde le document image '{nom_fichier_image}'.

TACHE :
"Qu'est-ce qui est marqué dans ce document ?"

Tu dois :
1. Identifier la nature du document.
2. Transcrire ce que tu arrives à lire (ingrédients, instructions...).
Base-toi sur le texte identifié comme [TEXTE MANUSCRIT].
"""

print(f"🎯 MISSION VISION : {mission_vision}")
try:
    res = agent.run(mission_vision)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission 12
# 1. On charge l'image pour que le module Vision l'analyse et la mette en mémoire
# ⚠️ REMPLACE PAR LE NOM EXACT DE TON FICHIER IMAGE ⚠️
nom_image = 'images.jpeg'
initialiser_rag([nom_image])

# 2. La Mission Multimodale
mission_12 = f"""
Agis comme un Expert en Présentation Visuelle.

TACHE 1 : ANALYSE
Interroge ta mémoire (RAG) pour savoir ce qui est décrit dans le document image '{nom_image}'.
Récupère une description claire de la scène.

TACHE 2 : CRÉATION POWERPOINT
Crée un fichier PowerPoint nommé 'analayse_image.pptx'.
1. Ajoute une nouvelle diapositive.
2. En TITRE de la diapositive, mets : "Analyse IA".
3. En CONTENU (Texte) de la diapositive, colle la description que tu as trouvée à l'étape 1.

TACHE 3 : INSERTION IMAGE
Sur cette MÊME diapositive (Slide 1), insère l'image originale '{nom_image}'.

Génère le fichier final.
"""

print(f"🎯 MISSION 12 : {mission_12}")
try:
    res = agent.run(mission_12)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 3 : RECHERCHE FINANCIÈRE COMPLEXE
fichiers_mission = ['Societe-Generale-Pilier-3_T2-2022_FR.pdf']
initialiser_rag(fichiers_mission)

mission_3 = """
Agis comme un Analyste Financier Spécialisé en Banque.
Analyse le document PDF 'Societe-Generale-Pilier-3_T2-2022_FR.pdf'.

TACHE :
Tu dois trouver le montant des fonds propres à la fin de la période (généralement le 30 juin 2022).
Cherche le tableau "KM1" (Key Metrics) et donne-moi les valeurs pour :
1. Les Fonds propres de base de catégorie 1 (CET1).
2. Les Fonds propres totaux.

Cite la page où se trouve l'information.
"""

print(f"🎯 MISSION 3 : {mission_3}")
try:
    res = agent.run(mission_3)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# --- TEST DIAGNOSTIC DE LA CLÉ ---
import google.generativeai as genai
import os

# Ta clé (elle est déjà en mémoire si tu as lancé les cellules précédentes)
if 'CLE_GOOGLE_VISION' in globals():
    genai.configure(api_key=CLE_GOOGLE_VISION)
    print(f"🔑 Clé détectée : {CLE_GOOGLE_VISION[:5]}...*****")
else:
    print("⚠️ Remets ta clé dans la variable CLE_GOOGLE_VISION")

try:
    # On essaie le modèle le plus récent pour voir si la clé répond
    print("📡 Tentative de contact direct avec Google (sans LangChain)...")

    # Test avec le modèle 'text-embedding-004' (le plus récent)
    result = genai.embed_content(
        model="models/text-embedding-004",
        content="Test de connexion",
        task_type="retrieval_document"
    )

    print("\n✅ SUCCÈS ! Ta clé fonctionne parfaitement.")
    print(f"   Google a renvoyé un vecteur de {len(result['embedding'])} chiffres.")
    print("   Conclusion : Le problème vient de LangChain, pas de ta clé.")

except Exception as e:
    print(f"\n❌ ÉCHEC. Message d'erreur : {e}")
    if "403" in str(e):
        print("   -> Là oui, c'est un problème de clé ou de permission.")
    elif "404" in str(e):
        print("   -> Le modèle demandé n'est pas dispo sur ton compte/région.")

In [ ]:
# --- DÉTECTIVE DES MODÈLES DISPONIBLES ---
import google.generativeai as genai
import os

# Ta clé (déjà en mémoire normalement)
if 'CLE_GOOGLE_VISION' in globals():
    genai.configure(api_key=CLE_GOOGLE_VISION)
else:
    print("⚠️ Clé introuvable, remets-la.")

print("🔍 Recherche des modèles d'Embedding disponibles pour ta clé...")

try:
    # On demande la liste de tous les modèles
    compteur = 0
    for m in genai.list_models():
        # On cherche ceux qui savent faire de l'embedding (embedContent)
        if 'embedContent' in m.supported_generation_methods:
            print(f"✅ TROUVÉ : {m.name}")
            compteur += 1

    if compteur == 0:
        print("❌ Aucun modèle d'embedding trouvé. C'est peut-être un souci de région (Europe).")
    else:
        print(f"\n👉 Copie le nom exact d'un des modèles ci-dessus (ex: models/embedding-001).")

except Exception as e:
    print(f"Erreur fatale : {e}")